# Correlation of Prediction Errors
We want to calculate the following values from Müller J (2021):

$$c_w=
\frac
{⟨(\vec{e}_i-⟨\vec{e}_i⟩_w) (\vec{e}_j-⟨\vec{e}_j⟩_w)⟩_w}
{\sigma_w(\vec{e}_i) \sigma_w(\vec{e}_j)}
$$

With:

* Prediction Error: $\vec e_i = | \vec p_i - \vec L |$
* Label (y_true): $\vec L$
* Weight factor: $w=\max \left(\vec{e}_i, \vec{e}_j\right) $
* Weighted Mean: $ ⟨z⟩_w=\frac{\sum_k w_k z_k}{\sum_k w_k} $
* Weighted Standard Deviation: $ \sigma_w(z)=\sqrt{⟨z^2⟩_w - ⟨ z⟩_w^2} $


In [1]:
from config import PATHS
import pandas as pd

from utils.io import pickle_path

In [2]:
pdir = PATHS.patient_dirs()[0]
pdir

PatientDir('/data/home/webb/UNEEG/datasets/competition/competition-01-MINIFAKE')

In [3]:
models = ['ensemble', 'CNN']

clips = pd.read_pickle(pickle_path(pdir.clips_table))
clips = clips[clips['valid']]
clips.head()

,start_seg,end_seg,end_time,ensemble_probability,CNN_probability,preictal,types,valid,segs_in_clip,full,n_existing,sufficient_data
1,18,57,2020-11-09 04:37:50.869500,0.640843,0.537007,False,[interictal],True,40,True,40,True
2,58,97,2020-11-09 04:47:50.779500,0.646217,0.684420,False,[interictal],True,40,True,40,True
3,98,137,2020-11-09 04:57:50.689500,0.629218,0.569690,False,[interictal],True,40,True,40,True
4,138,177,2020-11-09 05:07:50.599500,0.643703,0.674133,False,[interictal],True,40,True,40,True
5,178,217,2020-11-09 05:17:50.509500,0.634993,0.606353,False,[interictal],True,40,True,40,True


In [4]:
# Probabilities
p_cols = [f'{model}_probability' for model in models]
p = clips[p_cols] # result

In [5]:
p.head()

,ensemble_probability,CNN_probability
1,0.640843,0.537007
2,0.646217,0.684420
3,0.629218,0.569690
4,0.643703,0.674133
5,0.634993,0.606353


# Prediction Errors

In [6]:
# Labels
L = clips['preictal'].rename('L')
L.head()

1    False
2    False
3    False
4    False
5    False
Name: L, dtype: bool

In [14]:
# Prediction errors
e = p.subtract(L, axis=0).abs()
e.columns = [f'{m}_e' for m in models]
e.head()

,ensemble_e,CNN_e
1,0.640843,0.537007
2,0.646217,0.684420
3,0.629218,0.569690
4,0.643703,0.674133
5,0.634993,0.606353


In [19]:
# Weights
w = e.max(axis=1)
w.name = 'w'
w.head()

1    0.640843
2    0.684420
3    0.629218
4    0.674133
5    0.634993
Name: w, dtype: float32

In [22]:
# Weighted Means
def weighted_mean(w, x):
    return (w * x).sum() / w.sum()

# weighted means per column
means = e.mul(w, axis=0).sum(axis=0) / w.sum()
means.index = models
means

ensemble    0.569255
CNN         0.299172
dtype: float32

In [20]:
# Result
r = p.join(L).join(e).join(w)
r[r['L']].head()

,ensemble_probability,CNN_probability,L,ensemble_e,CNN_e,w
404,0.553064,0.386653,True,0.446936,0.613347,0.613347
405,0.611039,0.586521,True,0.388961,0.413479,0.413479
406,0.618376,0.628910,True,0.381624,0.371090,0.381624
407,0.638188,0.698275,True,0.361812,0.301725,0.361812
408,0.640150,0.645152,True,0.359850,0.354848,0.359850
